In [2]:
import copy
import json
import os
import sys
import warnings
import yaml

warnings.filterwarnings("ignore")

# =============================================================================
# 1. 프로젝트 Root 경로 및 Vector DB 경로 설정
# =============================================================================
PROJECT_ROOT = "/home/taehoon/sprint-ai-mid-project_team3"
VECTOR_DB_PATH = "/data/processed/vector_db/local"  # 지정된 Vector DB 경로

if os.path.abspath(PROJECT_ROOT) not in sys.path:
    sys.path.append(os.path.abspath(PROJECT_ROOT))

from src.retriever_factory import create_retriever

# 설정 파일(YAML) 로드 및 예외 처리
CONFIG_PATH = os.path.join(PROJECT_ROOT, "config/default.yaml")
if not os.path.exists(CONFIG_PATH):
    CONFIG_PATH = os.path.join(PROJECT_ROOT, "config.yaml")

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(f"설정 파일이 존재하지 않습니다: {CONFIG_PATH}")

try:
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        full_config = yaml.safe_load(f) or {}
except yaml.YAMLError as e:
    print(f"\n[🚨 YAML 파일 구문 에러 예외 발생] 경로: {CONFIG_PATH}")
    raise e

# Vector DB 저장 경로를 Config 내부에 반영
retrieval_config = full_config.get("retrieval") or full_config.get("retriever", {})
if "profiles" in retrieval_config and "local" in retrieval_config["profiles"]:
    retrieval_config["profiles"]["local"]["persist_directory"] = VECTOR_DB_PATH
    retrieval_config["profiles"]["local"]["db_path"] = VECTOR_DB_PATH

# =============================================================================
# 2. 실제 데이터 경로 연결 (data/processed/chunks_800_120.jsonl)
# =============================================================================
# [수정: os.path.join 내부 선두 슬래시 제거로 PROJECT_ROOT 경로 유지]
CHUNK_FILE_PATH = os.path.join(PROJECT_ROOT, "/data/processed/chunks_800_120.jsonl")

if not os.path.exists(CHUNK_FILE_PATH):
    raise FileNotFoundError(
        f"\n[🚨 FileNotFoundError]\n"
        f"지정한 실제 데이터 파일이 존재하지 않습니다: {CHUNK_FILE_PATH}\n"
        f"-> 터미널에서 청크 생성 스크립트(예: python -m scripts.build_chunks)를 실행해 주세요."
    )

chunks_for_factory = []
seen_doc_ids = set()

with open(CHUNK_FILE_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        clean_line = line.replace("\xa0", " ").strip()
        if not clean_line:
            continue

        try:
            item = json.loads(clean_line)

            text_content = item.get("text") or item.get("page_content") or item.get("content") or ""
            if not text_content.strip():
                continue

            item["text"] = text_content
            meta = item.get("metadata") if isinstance(item.get("metadata"), dict) else {}

            agency_val = (
                item.get("agency")
                or meta.get("agency")
                or meta.get("organization")
                or meta.get("발주기관")
                or "미지정"
            ).strip()

            title_val = (
                item.get("project_name")
                or meta.get("project_name")
                or meta.get("title")
                or meta.get("사업명")
                or "미지정"
            ).strip()

            doc_id_val = (
                item.get("doc_id")
                or meta.get("doc_id")
                or item.get("chunk_id")
                or meta.get("chunk_id")
                or f"DOC_{line_num}"
            )

            dedup_key = f"{doc_id_val}_{text_content[:30]}"
            if dedup_key in seen_doc_ids:
                continue
            seen_doc_ids.add(dedup_key)

            item["agency"] = agency_val
            item["project_name"] = title_val
            item["doc_id"] = doc_id_val

            meta["agency"] = agency_val
            meta["project_name"] = title_val
            meta["title"] = title_val
            meta["doc_id"] = doc_id_val
            item["metadata"] = meta

            chunks_for_factory.append(item)

        except json.JSONDecodeError as err:
            print(f"⚠️ {line_num}번째 줄 JSON 파싱 실패: {err}")

if not chunks_for_factory:
    raise ValueError(f"❌ '{CHUNK_FILE_PATH}' 파일에서 유효한 데이터를 읽지 못했습니다.")

print(f"✅ 총 {len(chunks_for_factory)}개의 실제 청크 데이터 로드 완료! (파일: {os.path.basename(CHUNK_FILE_PATH)})")

first_meta = chunks_for_factory[0].get("metadata", {})
real_agency_sample = first_meta.get("agency", "미지정")
print(f"   [로드 검증] 대표 기관: {real_agency_sample} | 사업명: {first_meta.get('project_name')}")
print(f"   [본문 시작]: {chunks_for_factory[0].get('text', '')[:50]}...\n")

# =============================================================================
# 3. 전체 컬렉션 DB 적재 및 통합 리트리버 생성
# =============================================================================
default_config = copy.deepcopy(full_config)
if "retrieval" not in default_config:
    default_config["retrieval"] = retrieval_config

ret_cfg = default_config["retrieval"]
if "profiles" in ret_cfg and "local" in ret_cfg["profiles"]:
    ret_cfg["profiles"]["local"]["collection_name"] = "total_rfp_collection"
    ret_cfg["profiles"]["local"]["persist_directory"] = VECTOR_DB_PATH

# [수정: 정상 규격대로 create_retriever 호출]
retriever = create_retriever(
    chunks=chunks_for_factory,
    retrieval_config=default_config,
    profile="local",
)
print(f"Chroma DB ({VECTOR_DB_PATH}) 적재 완료!\n")

# =============================================================================
# 4. 단일 컬렉션 대상 조건별 검색 검증
# =============================================================================
query_str = "사업의 추진 배경 및 기대 효과를 알려주세요"

print("=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===")
results = retriever.search(query=query_str, top_k=3)

printed_ids = set()
for d in results:
    doc_id = d.metadata.get("doc_id", "N/A")
    agency_name = d.metadata.get("agency", "N/A")
    score_val = getattr(d, "score", 0.0)

    if doc_id in printed_ids:
        continue
    printed_ids.add(doc_id)
    print(f"[{doc_id} / {agency_name}](Score: {score_val:.4f}) {d.text[:60]}...")

# [수정: filters={"agency": ...} 전달 규격 맞춤]
print(f"\n=== 2. 실제 기관명 필터 적용 ('{real_agency_sample}' 필터링) ===")
results_filtered = retriever.search(
    query=query_str,
    top_k=2,
    filters={"agency": real_agency_sample}
)
for d in results_filtered:
    agency_name = d.metadata.get("agency", "N/A")
    project_name = d.metadata.get("project_name", "N/A")
    score_val = getattr(d, "score", 0.0)
    print(f"[{agency_name} / {project_name}](Score: {score_val:.4f}) {d.text[:60]}...")

# =============================================================================
# 5. Multi-Collection 하이브리드 검증
# =============================================================================
print("\n" + "=" * 50)
print("=== 5. Multi-Collection 실제 데이터 하이브리드 검증 ===")
print("=" * 50)

target_agency_clean = real_agency_sample.strip()

filtered_chunks = []
for c in chunks_for_factory:
    c_agency = str(c.get("agency", ""))
    m_agency = str(c.get("metadata", {}).get("agency", ""))
    
    if (target_agency_clean in c_agency) or (target_agency_clean in m_agency) or (c_agency == "미지정"):
        filtered_chunks.append(c)

if len(filtered_chunks) == 0:
    filtered_chunks = chunks_for_factory[:100]

print(f"-> Multi-Collection 대상 필터링된 청크 수: {len(filtered_chunks)}개")

msit_run_config = copy.deepcopy(default_config)
msit_ret_cfg = msit_run_config["retrieval"]

if "profiles" in msit_ret_cfg and "local" in msit_ret_cfg["profiles"]:
    msit_ret_cfg["profiles"]["local"]["collection_name"] = "filtered_actual_collection"
    msit_ret_cfg["profiles"]["local"]["persist_directory"] = VECTOR_DB_PATH

agency_retriever = create_retriever(
    chunks=filtered_chunks,
    retrieval_config=msit_run_config,
    profile="local"
)

hybrid_query = "사업 추진 배경"
print(f"-> '{target_agency_clean}' 전용 컬렉션 대상 검색 질의: '{hybrid_query}'")

results_hybrid = agency_retriever.search(query=hybrid_query, top_k=2)

print(f"Multi-Collection 하이브리드 테스트 완료 (반환된 결과: {len(results_hybrid)}건)")
printed_hybrid_ids = set()
for idx, d in enumerate(results_hybrid):
    p_name = d.metadata.get("project_name", "N/A")
    doc_id = d.metadata.get("doc_id", f"HYBRID_{idx+1}")
    
    if doc_id in printed_hybrid_ids:
        continue
    printed_hybrid_ids.add(doc_id)
    
    print(f"   [{idx+1}] {p_name} ({doc_id}) -> {d.text[:50]}...")

✅ 총 10215개의 실제 청크 데이터 로드 완료! (파일: chunks_800_120.jsonl)
   [로드 검증] 대표 기관: 한영대학 | 사업명: 한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화
   [본문 시작]: 2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 고도화 제안요청서 202...



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chroma DB (/data/processed/vector_db/local) 적재 완료!

=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===
[DOC_RFP_002 / 과학기술정보통신부](Score: 0.2384) 1. 사업 개요...
[DOC_RFP_003 / 산업통상자원부](Score: 1.1448) 사업명은 "2026년 공공 AI 학습지원 플랫폼 구축 사업"이며 발주기관은 가상디지털진흥원이다. 사업 목적은...

=== 2. 실제 기관명 필터 적용 ('한영대학' 필터링) ===
안내: '{'agency': '한영대학'}' 필터 조건에 일치하는 문서가 후보군(100개) 내에 없습니다.

=== 5. Multi-Collection 실제 데이터 하이브리드 검증 ===
-> Multi-Collection 대상 필터링된 청크 수: 61개


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

-> '한영대학' 전용 컬렉션 대상 검색 질의: '사업 추진 배경'
Multi-Collection 하이브리드 테스트 완료 (반환된 결과: 0건)
